# Домашняя работа: квантизация и дистилляция LLM

В этой работе вы самостоятельно проведёте эксперименты с квантизацией и дистилляцией LLM и проверите, сколько качества при этом теряет модель. Задача, на которую обучаемся, — разбор запросов к голосовому ассистенту из датасета MASSIVE (ru-RU): по запросу модель возвращает JSON с intent и slots. Подробнее: https://huggingface.co/datasets/AmazonScience/massive

### Сперва настроим окружение и подготовим код для экспериментов

In [ ]:
# Раскомментируйте в новом окружении, затем перезапустите kernel:
# %pip install "torch==2.8.0" "transformers==4.55.2" "datasets==4.1.1" \
#     "accelerate==1.10.1" "llmcompressor==0.8.1" "compressed-tensors==0.12.2" \
#     "numpy==2.2.6" "pandas==2.2.3" "safetensors>=0.5" "huggingface-hub<1" tqdm

In [1]:
import gc
import json
import math
import random
import re
import shutil
import struct
from collections import Counter
from collections.abc import Sequence
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from importlib.metadata import version
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import Dataset, DatasetDict, load_dataset
from IPython.display import display
from tqdm.auto import tqdm
from transformers import (
    AutoConfig, AutoModelForCausalLM, AutoTokenizer,
    CompressedTensorsConfig, GenerationConfig, PreTrainedModel, PreTrainedTokenizerBase,
    Trainer, TrainingArguments, set_seed,
)

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 42
ARTIFACTS = Path("artifacts_compression_hw")
ARTIFACTS.mkdir(exist_ok=True)
set_seed(SEED)

def clean_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def ensure_empty_dir(path: str | Path) -> Path:
    path = Path(path)
    if path.exists() and any(path.iterdir()):
        raise FileExistsError(f"{path} уже существует и не пустой")
    path.mkdir(parents=True, exist_ok=True)
    return path

environment = {
    p: version(p) for p in (
        "torch", "transformers", "datasets", "llmcompressor",
        "compressed-tensors", "numpy", "safetensors",
    )
}
print(environment)
(ARTIFACTS / "environment.json").write_text(
    json.dumps(environment, indent=2), encoding="utf-8"
)

{'torch': '2.8.0+cu126', 'transformers': '4.55.2', 'datasets': '4.1.1', 'llmcompressor': '0.8.1', 'compressed-tensors': '0.12.2', 'numpy': '2.2.6', 'safetensors': '0.8.0'}


187

## Загрузка MASSIVE

Используем датасет MASSIVE. Все калибровочные примеры и данные для дистилляции берём только из train. Оцениваем на валидации (dev).

In [3]:
DATASET_ID = "AmazonScience/massive"
DATASET_REVISION = "ed58ac423a2f4121720918bf5301577edce4ffd3"  # refs/convert/parquet
LOCALE = "ru-RU"

dataset_url = f"hf://datasets/{DATASET_ID}@{DATASET_REVISION}/{LOCALE}"
data_files = {
    "train": f"{dataset_url}/train/0000.parquet",
    "validation": f"{dataset_url}/validation/0000.parquet",
    "test": f"{dataset_url}/test/0000.parquet",
}
raw = load_dataset("parquet", data_files=data_files)
print(raw)

DatasetDict({
    train: Dataset({
        features: ['id', 'locale', 'partition', 'scenario', 'intent', 'utt', 'annot_utt', 'worker_id', 'slot_method', 'judgments'],
        num_rows: 11514
    })
    validation: Dataset({
        features: ['id', 'locale', 'partition', 'scenario', 'intent', 'utt', 'annot_utt', 'worker_id', 'slot_method', 'judgments'],
        num_rows: 2033
    })
    test: Dataset({
        features: ['id', 'locale', 'partition', 'scenario', 'intent', 'utt', 'annot_utt', 'worker_id', 'slot_method', 'judgments'],
        num_rows: 2974
    })
})


In [4]:
example_index = 0
example_raw = raw["train"][example_index]
print("Исходная запись MASSIVE:")
print(json.dumps(
    {key: example_raw[key] for key in ("id", "utt", "annot_utt", "intent")},
    ensure_ascii=False, indent=2,
))

Исходная запись MASSIVE:
{
  "id": "1",
  "utt": "разбуди меня в девять утра в пятницу",
  "annot_utt": "разбуди меня в [time : девять утра] в [date : пятницу]",
  "intent": 48
}


### Сконвертируем данные для калибровки

В MASSIVE текст запроса лежит в поле `utt`, intent записан числом, а slots отмечены прямо в тексте поля `annot_utt`, например `[time : семь утра]`. Модель должна по запросу вернуть JSON с названием intent и списком slots в виде пар «тип — значение». Поэтому преобразуем запись в `text` — исходный запрос — и `gold` — эталонный JSON-ответ. Такой формат удобнее для калибровки и подсчёта метрик.

In [5]:
SLOT_PATTERN = re.compile(r"\[([^:\[\]]+?)\s*:\s*([^\[\]]*?)\]")

def extract_slots(annot_utt: str) -> list[list[str]]:
    """Достаёт slots из разметки MASSIVE вида `[time : семь утра]`.

    Args:
        annot_utt: размеченный запрос из поля `annot_utt`.

    Returns:
        Список пар `[тип, значение]` в порядке появления в запросе.
    """
    return [[m.group(1).strip(), m.group(2).strip()]
            for m in SLOT_PATTERN.finditer(annot_utt)]

def target_json(intent: str, slots: list[list[str]]) -> str:
    """Собирает эталонный ответ в компактный JSON.

    Args:
        intent: название intent.
        slots: список пар `[тип, значение]`.

    Returns:
        Строка `{"intent":...,"slots":[...]}` без пробелов между полями.
    """
    return json.dumps(
        {"intent": intent, "slots": slots},
        ensure_ascii=False, separators=(",", ":"),
    )

def prepare_massive(raw_splits: DatasetDict) -> DatasetDict:
    """Переводит сплиты MASSIVE в формат запрос - эталонный JSON.

    Args:
        raw_splits: исходные сплиты MASSIVE с полями `id`, `utt`, `intent`, `annot_utt`.

    Returns:
        Сплиты с полями `id`, `text` (запрос) и `gold` (эталонный JSON).
    """
    intent_feature = raw_splits["train"].features["intent"]

    def convert(row: dict[str, Any]) -> dict[str, str]:
        intent = intent_feature.int2str(row["intent"])
        slots = extract_slots(row["annot_utt"])
        return {
            "id": str(row["id"]), "text": row["utt"],
            "gold": target_json(intent, slots),
        }

    return DatasetDict({
        split: ds.map(convert, remove_columns=ds.column_names)
        for split, ds in raw_splits.items()
    })

In [6]:

data = prepare_massive(raw)
INTENTS = list(raw["train"].features["intent"].names)
SLOT_TYPES = sorted({
    kind for row in raw["train"]
    for kind, value in extract_slots(row["annot_utt"])
})
train_data = data["train"]
DEV_SAMPLES = 256
dev_data = data["validation"]
if DEV_SAMPLES is not None:
    dev_data = dev_data.shuffle(seed=SEED).select(range(min(DEV_SAMPLES, len(dev_data))))
print("Число классов:", len(INTENTS), "типов slots:", len(SLOT_TYPES))

Число классов: 60 типов slots: 54


In [7]:
example_prepared = train_data[example_index]
print("Запрос модели (text):")
print(example_prepared["text"])
print("\nЭталонный ответ (gold):")
print(json.dumps(json.loads(example_prepared["gold"]), ensure_ascii=False, indent=2))

Запрос модели (text):
разбуди меня в девять утра в пятницу

Эталонный ответ (gold):
{
  "intent": "alarm_set",
  "slots": [
    [
      "time",
      "девять утра"
    ],
    [
      "date",
      "пятницу"
    ]
  ]
}


### Генерация ответов
Всем моделям даём одинаковую инструкцию и одинаковые описания допустимых классов. Генерация детерминирована: greedy decoding, одинаковый лимит токенов.

In [8]:
BASE_MODEL_ID = "Qwen/Qwen3-8B" # можно пробовать модели того же семейства, но поменьше, чтобы считалось быстрее

# Значения используются как пример, при необходимости можно регулировать
MAX_PROMPT_TOKENS = 2048
MAX_NEW_TOKENS = 128
GENERATE_BATCH_SIZE = 8

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda:0


In [9]:
system_prompt = (
    'Определи intent и извлеки slots. Верни только JSON вида '
    '{"intent":"...","slots":[["тип","значение"],...]}. '
    'Значения slots копируй из запроса. Если slots нет, верни пустой список. '
    "Не добавляй объяснения.\nДопустимые intents: " + ", ".join(INTENTS)
    + "\nДопустимые типы slots: " + ", ".join(SLOT_TYPES)
)

def encode_prompt(tokenizer: PreTrainedTokenizerBase, text: str) -> list[int]:
    """Токенизирует запрос в chat-формате с общим system prompt.

    Args:
        tokenizer: токенизатор модели с chat template.
        text: запрос пользователя.

    Returns:
        Token ids промпта, заканчивающиеся началом ответа ассистента.
    """
    ids = tokenizer.apply_chat_template(
        [{"role": "system", "content": system_prompt},
         {"role": "user", "content": text}],
        tokenize=True, add_generation_prompt=True, enable_thinking=False,
    )
    if len(ids) > MAX_PROMPT_TOKENS:
        raise ValueError(f"Prompt имеет {len(ids)} токенов. Увеличьте лимит, не обрезайте запрос.")
    return ids

def generation_config(tokenizer: PreTrainedTokenizerBase) -> GenerationConfig:
    pad_token_id = tokenizer.pad_token_id
    if pad_token_id is None:
        pad_token_id = tokenizer.eos_token_id
    return GenerationConfig(
        do_sample=False, num_beams=1, max_new_tokens=MAX_NEW_TOKENS,
        eos_token_id=tokenizer.eos_token_id, pad_token_id=pad_token_id,
        use_cache=True,
    )

def trim_answer(token_ids: list[int], eos_token_id: int, pad_token_id: int) -> list[int]:
    """Отрезает паддинг, после генерации

    Returns:
        Токены ответа; eos остаётся последним, если модель его сгенерировала.
    """
    answer = []
    for token_id in token_ids:
        answer.append(token_id)
        if token_id == eos_token_id:
            break
        if token_id == pad_token_id:
            answer.pop()
            break
    return answer

@torch.inference_mode()
def generate_answer_ids(
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizerBase,
    prompts: Sequence[list[int]],
) -> list[list[int]]:
    """Жадно генерирует ответы на батч промптов.

    Args:
        model: causal LM.
        tokenizer: токенизатор модели.
        prompts: token ids промптов; паддинг добавляется слева.

    Returns:
        Токены ответа для каждого промпта в том же порядке, без промпта и паддинга.
    """
    config = generation_config(tokenizer)
    width = max(len(ids) for ids in prompts)
    input_ids, attention_mask = [], []

    for ids in prompts:
        padding = width - len(ids)
        input_ids.append([config.pad_token_id] * padding + list(ids))
        attention_mask.append([0] * padding + [1] * len(ids))
    input_ids = torch.tensor(input_ids, dtype=torch.long, device=model.device)
    attention_mask = torch.tensor(attention_mask, dtype=torch.long, device=model.device)

    output = model.generate(
        input_ids=input_ids, attention_mask=attention_mask,
        generation_config=config, do_sample=False, num_beams=1,
    )
    return [
        trim_answer(row, config.eos_token_id, config.pad_token_id)
        for row in output[:, input_ids.shape[1]:].tolist()
    ]

In [10]:
BASELINE_PATH = ARTIFACTS / "baseline"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
baseline_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, torch_dtype=torch.bfloat16,
    device_map=DEVICE,
).eval()
ensure_empty_dir(BASELINE_PATH)
baseline_model.save_pretrained(BASELINE_PATH, safe_serialization=True)
tokenizer.save_pretrained(BASELINE_PATH)
print("Исходная модель:", BASE_MODEL_ID)
print("Пример prompt, токенов:", len(encode_prompt(tokenizer, train_data[0]["text"])))

Loading checkpoint shards: 100%|██████████| 5/5 [00:04<00:00,  1.06it/s]


FileExistsError: artifacts_compression_hw/baseline уже существует и не пустой

## Метрика качества

$$
Q = \tfrac{1}{2} \operatorname{IntentAccuracy} + \tfrac{1}{2} \operatorname{SlotMicroF1}.
$$

Slots сравниваются как **мультимножество** пар `(тип, значение)`: порядок не влияет, лишние дубликаты дают false positives. Неверный JSON или нарушение схемы считается ошибкой всего ответа.

Дополнительно считаем `frame_exact_match` и `valid_json_rate`. Если во всей выборке нет ни одного gold/predicted slot, slot F1 равен 1 только при валидности всех ответов, иначе 0.

In [12]:
def normalize_value(value: str) -> str:
    """Приводит значение slot к виду для сравнения."""
    return " ".join(value.casefold().split())

def parse_answer(
    text: str, intents: Sequence[str], slot_types: Sequence[str],
) -> dict[str, Any] | None:
    """Разбирает ответ модели и проверяет формат.

    Args:
        text: сырой ответ модели.
        intents: допустимые названия intent.
        slot_types: допустимые типы slots.

    Returns:
        Словарь с ключами `intent` и `slots` или None, если ответ невалиден.
    """
    try:
        obj = json.loads(text.strip())
        if not isinstance(obj, dict) or set(obj) != {"intent", "slots"}:
            return None
        if not isinstance(obj["intent"], str) or obj["intent"] not in intents:
            return None
        if not isinstance(obj["slots"], list):
            return None
        for slot in obj["slots"]:
            if not isinstance(slot, list) or len(slot) != 2:
                return None
            kind, value = slot
            if (not isinstance(kind, str) or kind not in slot_types
                    or not isinstance(value, str) or not value.strip()):
                return None
        return obj
    except (ValueError, TypeError):
        return None

def slot_counter(obj: dict[str, Any]) -> Counter:
    """Превращает slots ответа в мультимножество

    Args:
        obj: разобранный ответ с ключом `slots`.

    Returns:
        Counter пар `(тип, нормализованное значение)`.
    """
    return Counter((kind, normalize_value(value)) for kind, value in obj["slots"])

def compute_metrics(
    predictions: Sequence[str],
    golds: Sequence[str],
    intents: Sequence[str],
    slot_types: Sequence[str],
) -> dict[str, float]:
    """Считает метрики качества по ответам модели.

    Args:
        predictions: ответы модели.
        golds: эталонные JSON в том же порядке.
        intents: допустимые названия intent.
        slot_types: допустимые типы slots.

    Returns:
        Словарь с `quality`, `intent_accuracy`, `slot_micro_f1`,
        `frame_exact_match`, `valid_json_rate` и числом примеров `n`.
    """
    if len(predictions) != len(golds) or not golds:
        raise ValueError("Нужны непустые списки одинаковой длины.")
    correct_intent = exact = valid = tp = fp = fn = 0
    for text, gold_text in zip(predictions, golds):
        gold = parse_answer(gold_text, intents, slot_types)
        if gold is None:
            raise ValueError("Невалидная gold-разметка")
        pred = parse_answer(text, intents, slot_types)
        gold_slots = slot_counter(gold)
        pred_slots = Counter() if pred is None else slot_counter(pred)
        tp += sum((pred_slots & gold_slots).values())
        fp += sum((pred_slots - gold_slots).values())
        fn += sum((gold_slots - pred_slots).values())
        if pred is not None:
            valid += 1
            intent_ok = pred["intent"] == gold["intent"]
            correct_intent += int(intent_ok)
            exact += int(intent_ok and pred_slots == gold_slots)
    n = len(golds)
    denom = 2 * tp + fp + fn
    slot_f1 = 2 * tp / denom if denom else float(valid == n)
    intent_acc = correct_intent / n
    return {
        "quality": 0.5 * (intent_acc + slot_f1),
        "intent_accuracy": intent_acc,
        "slot_micro_f1": slot_f1,
        "frame_exact_match": exact / n,
        "valid_json_rate": valid / n,
        "n": n,
    }

def evaluate(
    model: PreTrainedModel, tokenizer: PreTrainedTokenizerBase, ds: Dataset,
    batch_size: int = GENERATE_BATCH_SIZE,
) -> tuple[dict[str, float], list[str]]:
    """Генерирует ответы на выборку батчами и считает метрики.

    Args:
        model: оцениваемая модель.
        tokenizer: её токенизатор.
        ds: выборка с полями `text` и `gold`.
        batch_size: число одновременных генераций.

    Returns:
        Метрики из compute_metrics и список ответов модели в порядке выборки.
    """
    model.eval()
    texts = ds["text"]
    predictions = []

    for start in tqdm(range(0, len(texts), batch_size), desc="Оценка"):
        batch_texts = texts[start:start + batch_size]
        prompts = [encode_prompt(tokenizer, text) for text in batch_texts]
        answer_ids = generate_answer_ids(model, tokenizer, prompts)
        predictions.extend(tokenizer.batch_decode(answer_ids, skip_special_tokens=True))

    metrics = compute_metrics(predictions, ds["gold"], INTENTS, SLOT_TYPES)
    return metrics, predictions

def evaluate_checkpoint(
    model_path: str | Path, ds: Dataset,
) -> tuple[dict[str, float], list[str]]:
    """Загружает сохранённый checkpoint, оценивает его и выгружает из памяти.

    Args:
        model_path: каталог dense или квантизированного checkpoint.
        ds: выборка с полями `text` и `gold`.

    Returns:
        Метрики и ответы модели, как в evaluate.
    """
    model_path = str(model_path)
    model = AutoModelForCausalLM.from_pretrained(
        model_path, torch_dtype=torch.bfloat16,
        device_map=DEVICE,
    )
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        return evaluate(model, tokenizer, ds)
    finally:
        del model
        clean_memory()

Посчитаем метрики бейзлайна.

In [12]:
baseline_metrics, baseline_predictions = evaluate(baseline_model, tokenizer, dev_data)
print(baseline_metrics)
(ARTIFACTS / "baseline_dev.json").write_text(
    json.dumps(baseline_metrics, indent=2), encoding="utf-8"
)
results = [{
    "run": "baseline", "path": str(BASELINE_PATH),
    "method": "dense", "calibration": None, **baseline_metrics,
}]
del baseline_model
clean_memory()

Оценка:   0%|          | 0/32 [00:00<?, ?it/s]`generation_config` default values have been modified to match model-specific defaults: {'do_sample': True, 'temperature': 0.6, 'top_k': 20, 'top_p': 0.95, 'bos_token_id': 151643}. If this is not desired, please set these values explicitly.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Оценка: 100%|██████████| 32/32 [00:59<00:00,  1.87s/it]


{'quality': 0.28351873536299765, 'intent_accuracy': 0.375, 'slot_micro_f1': 0.1920374707259953, 'frame_exact_match': 0.10546875, 'valid_json_rate': 0.5234375, 'n': 256}


## Квантизация (5 баллов)

Используя код с семинара выполните следующие задания:
1) На примере одного из методов откалибруйте бейзлайн на MASSIVE train и на WikiText, сравните качество на dev. Есть ли влияние калибровочного датасета на метрику? (1 балл)

2) Поэкспериментируйте с разными конфигурациями методов AWQ, GPTQ, SmoothQuant (3 балла)

    Запустите минимум три конфигурации, пробуйте менять разные факторы:

    - битность
    - число калибровочных примеров;
    - group size
    - symmetric / asymmetric quantization;
    - smoothing_strength
    - Дополнительно (+1 балл) — какие слои оставить в исходной точности

3) Объясните результат — какие параметры наиболее значимы? (1 балл)


In [13]:
from llmcompressor import oneshot
from llmcompressor.modifiers.awq import AWQModifier
from llmcompressor.modifiers.quantization import GPTQModifier
from llmcompressor.modifiers.smoothquant import SmoothQuantModifier

CALIBRATION_MAX_SEQ_LENGTH = 1024

QUANT_EXPERIMENTS = {
    # Сравнение калибровочных данных.
    'awq_w4a16_massive_256': {
        'method': 'awq', 'scheme': 'W4A16',
        'calibration': 'massive', 'num_samples': 256, 'ignore': [],
    },
    'awq_w4a16_wikitext_256': {
        'method': 'awq', 'scheme': 'W4A16',
        'calibration': 'wikitext', 'num_samples': 256, 'ignore': [],
    },
    # Сравнение размера task-specific калибровки.
    'awq_w4a16_massive_64': {
        'method': 'awq', 'scheme': 'W4A16',
        'calibration': 'massive', 'num_samples': 64, 'ignore': [],
    },
    # Сравнение алгоритма и битности при одинаковой калибровке.
    'gptq_w4a16_massive_256': {
        'method': 'gptq', 'scheme': 'W4A16',
        'calibration': 'massive', 'num_samples': 256, 'ignore': [],
    },
    'gptq_w3a16_massive_256': {
        'method': 'gptq', 'scheme': 'W3A16',
        'calibration': 'massive', 'num_samples': 256, 'ignore': [],
    },
    # Сравнение силы сглаживания SmoothQuant.
    'smoothquant_w8a8_a05_massive_256': {
        'method': 'smoothquant', 'scheme': 'W8A8',
        'calibration': 'massive', 'num_samples': 256,
        'smoothing_strength': 0.5, 'ignore': [],
    },
    'smoothquant_w8a8_a08_massive_256': {
        'method': 'smoothquant', 'scheme': 'W8A8',
        'calibration': 'massive', 'num_samples': 256,
        'smoothing_strength': 0.8, 'ignore': [],
    },
    #тот же AWQ, но выходная голова остаётся в bf16.
    'awq_w4a16_massive_256_lm_head_bf16': {
        'method': 'awq', 'scheme': 'W4A16',
        'calibration': 'massive', 'num_samples': 256, 'ignore': ['lm_head'],
    },
}

def render_prompt(tokenizer: PreTrainedTokenizerBase, text: str) -> str:
    return tokenizer.apply_chat_template(
        [{'role': 'system', 'content': system_prompt},
         {'role': 'user', 'content': text}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )

def massive_calibration_dataset(
    tokenizer: PreTrainedTokenizerBase, num_samples: int,
) -> Dataset:
    rows = train_data.shuffle(seed=SEED)
    prompts = []
    for row in rows:
        prompt = render_prompt(tokenizer, row['text'])
        token_count = len(tokenizer(prompt, add_special_tokens=False)['input_ids'])
        if token_count <= CALIBRATION_MAX_SEQ_LENGTH:
            prompts.append(prompt)
        if len(prompts) == num_samples:
            break
    if len(prompts) != num_samples:
        raise ValueError('В MASSIVE train недостаточно промптов нужной длины.')
    return Dataset.from_dict({'text': prompts})

def wikitext_calibration_dataset(num_samples: int) -> Dataset:
    wiki = load_dataset('wikitext', 'wikitext-2-v1', split='train')
    candidates = [
        text for text in wiki['text']
        if text.strip() and len(text.split()) > 20
    ]
    rng = random.Random(SEED)
    texts = rng.sample(candidates, num_samples)
    return Dataset.from_dict({'text': texts})

def calibration_dataset(
    kind: str, tokenizer: PreTrainedTokenizerBase, num_samples: int,
) -> Dataset:
    if kind == 'massive':
        return massive_calibration_dataset(tokenizer, num_samples)
    if kind == 'wikitext':
        return wikitext_calibration_dataset(num_samples)
    raise ValueError(f'Неизвестный калибровочный датасет: {kind}')

def build_recipe(config: dict[str, Any]) -> list[Any]:
    ignore = config.get('ignore', [])
    method = config['method']
    if method == 'awq':
        return [AWQModifier(
            scheme=config['scheme'], targets=['Linear'], ignore=ignore,
        )]
    if method == 'gptq':
        return [GPTQModifier(
            scheme=config['scheme'], targets='Linear', ignore=ignore,
        )]
    if method == 'smoothquant':
        return [
            SmoothQuantModifier(
                smoothing_strength=config['smoothing_strength'],
            ),
            GPTQModifier(
                scheme=config['scheme'], targets='Linear', ignore=ignore,
            ),
        ]
    raise ValueError(f'Неизвестный метод: {method}')

def save_predictions(
    path: Path, rows: Dataset, predictions: Sequence[str],
) -> None:
    records = [
        {'id': row['id'], 'text': row['text'], 'gold': row['gold'],
         'prediction': prediction}
        for row, prediction in zip(rows, predictions)
    ]
    path.write_text(
        ''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in records),
        encoding='utf-8',
    )

def run_quantization_experiment(name: str) -> dict[str, float]:
    if name not in QUANT_EXPERIMENTS:
        raise KeyError(f'Нет конфигурации {name}.')
    config = QUANT_EXPERIMENTS[name]
    run_dir = ARTIFACTS / name
    model_dir = ensure_empty_dir(run_dir / 'model')
    (run_dir / 'config.json').write_text(
        json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8',
    )

    run_tokenizer = AutoTokenizer.from_pretrained(BASELINE_PATH)
    calibration_ds = calibration_dataset(
        config['calibration'], run_tokenizer, config['num_samples'],
    )
    model = AutoModelForCausalLM.from_pretrained(
        BASELINE_PATH, torch_dtype=torch.bfloat16, device_map=DEVICE,
    ).eval()
    try:
        oneshot(
            model=model, tokenizer=run_tokenizer, dataset=calibration_ds,
            recipe=build_recipe(config),
            max_seq_length=CALIBRATION_MAX_SEQ_LENGTH,
            num_calibration_samples=len(calibration_ds),
        )
        model.save_pretrained(model_dir, save_compressed=True)
        run_tokenizer.save_pretrained(model_dir)
    finally:
        del model
        clean_memory()

    metrics, predictions = evaluate_checkpoint(model_dir, dev_data)
    (run_dir / 'metrics.json').write_text(
        json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8',
    )
    save_predictions(run_dir / 'predictions.jsonl', dev_data, predictions)

    row = {'run': name, 'path': str(model_dir), **config, **metrics}
    global results
    results = [old for old in results if old['run'] != name] + [row]
    pd.DataFrame(results).to_json(
        ARTIFACTS / 'quantization_results.json',
        orient='records', force_ascii=False, indent=2,
    )
    display(pd.DataFrame(results))
    return metrics

def evaluate_saved_quantization_experiment(name: str) -> dict[str, float]:
    if name not in QUANT_EXPERIMENTS:
        raise KeyError(f'Нет конфигурации {name}.')
    config = QUANT_EXPERIMENTS[name]
    run_dir = ARTIFACTS / name
    model_dir = run_dir / 'model'
    if not model_dir.is_dir():
        raise FileNotFoundError(f'Не найден checkpoint: {model_dir}')
    metrics, predictions = evaluate_checkpoint(model_dir, dev_data)
    (run_dir / 'metrics.json').write_text(
        json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8',
    )
    save_predictions(run_dir / 'predictions.jsonl', dev_data, predictions)
    row = {'run': name, 'path': str(model_dir), **config, **metrics}
    global results
    results = [old for old in results if old['run'] != name] + [row]
    pd.DataFrame(results).to_json(
        ARTIFACTS / 'quantization_results.json',
        orient='records', force_ascii=False, indent=2,
    )
    display(pd.DataFrame(results))
    return metrics

In [14]:
run_quantization_experiment('awq_w4a16_massive_256')

Tokenizing: 100%|██████████| 256/256 [00:00<00:00, 705.98 examples/s]

2026-11-26T16:34:18.648273+0000 | reset | INFO - Compression lifecycle reset
2026-11-26T16:34:18.655146+0000 | _create_default_logger | INFO - Logging all LLM Compressor modifier-level logs to sparse_logs/26-11-2026_16.34.18.log
2026-11-26T16:34:18.655960+0000 | from_modifiers | INFO - Creating recipe from modifiers


2026-11-26T16:34:18.708272+0000 | on_initialize | INFO - No AWQModifier.mappings provided, inferring from model...


Resolving mapping 1/4 (0 skipped): : 36it [00:00, 2093.84it/s]
Resolving mapping 2/4 (35 skipped): : 36it [00:00, 2826.88it/s]
Resolving mapping 3/4 (0 skipped): : 36it [00:00, 2249.33it/s]
Resolving mapping 4/4 (0 skipped): : 36it [00:00, 2885.82it/s]

2026-11-26T16:34:18.778410+0000 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-11-26T16:34:18.778859+0000 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `AWQModifier`



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The

2026-11-26T16:52:05.163835+0000 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2026-11-26T16:52:05.228860+0000 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`
2026-11-26T16:52:05.229704+0000 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.



Compressing model: 253it [00:09, 25.79it/s]
/opt/conda/lib/python3.11/site-packages/transformers/quantizers/auto.py:231: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.However, loading attributes (e.g. ['run_compressed']) will be overwritten with the one you passed to `from_pretrained`. The rest will be ignored.
  warnings.warn(warning_msg)
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  5.81it/s]
Decompressing model: 253it [00:19, 12.74it/s]
Оценка: 100%|██████████| 32/32 [06:46<00:00, 12.72s/it]


,run,path,method,calibration,quality,intent_accuracy,slot_micro_f1,frame_exact_match,valid_json_rate,n,scheme,num_samples,ignore
0,baseline,artifacts_compression_hw/baseline,dense,None,0.283519,0.375,0.192037,0.105469,0.523438,256,NaN,NaN,NaN
1,awq_w4a16_massive_256,artifacts_compression_hw/awq_w4a16_massive_256...,awq,massive,0.345517,0.500,0.191033,0.070312,0.699219,256,W4A16,256.0,[]


{'quality': 0.3455165692007797,
 'intent_accuracy': 0.5,
 'slot_micro_f1': 0.19103313840155944,
 'frame_exact_match': 0.0703125,
 'valid_json_rate': 0.69921875,
 'n': 256}

In [15]:
run_quantization_experiment('awq_w4a16_wikitext_256')

Tokenizing: 100%|██████████| 256/256 [00:00<00:00, 1074.57 examples/s]

2026-11-26T17:05:50.290728+0000 | reset | INFO - Compression lifecycle reset
2026-11-26T17:05:50.293766+0000 | from_modifiers | INFO - Creating recipe from modifiers
2026-11-26T17:05:50.345411+0000 | on_initialize | INFO - No AWQModifier.mappings provided, inferring from model...



Resolving mapping 1/4 (0 skipped): : 36it [00:00, 1999.51it/s]
Resolving mapping 2/4 (35 skipped): : 36it [00:00, 2778.45it/s]
Resolving mapping 3/4 (0 skipped): : 36it [00:00, 2310.03it/s]
Resolving mapping 4/4 (0 skipped): : 36it [00:00, 2827.78it/s]

2026-11-26T17:05:50.416844+0000 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-11-26T17:05:50.417369+0000 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `AWQModifier`



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The

2026-11-26T17:23:11.990826+0000 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2026-11-26T17:23:12.055816+0000 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`
2026-11-26T17:23:12.056687+0000 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 253it [00:10, 24.45it/s]
/opt/conda/lib/python3.11/site-packages/transformers/quantizers/auto.py:231: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.However, loading attributes (e.g. ['run_compressed']) will be overwritten with the one you passed to `from_pretrained`. The rest will be ignored.
  warnings.warn(warning_msg)
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  8.81it/s]
Decompressing model: 253it [00:22, 11.43it/s]
Оценка: 100%|██████████| 32/32 [06:08<00:00, 11.53s/it]


,run,path,method,calibration,quality,intent_accuracy,slot_micro_f1,frame_exact_match,valid_json_rate,n,scheme,num_samples,ignore
0,baseline,artifacts_compression_hw/baseline,dense,None,0.283519,0.375000,0.192037,0.105469,0.523438,256,NaN,NaN,NaN
1,awq_w4a16_massive_256,artifacts_compression_hw/awq_w4a16_massive_256...,awq,massive,0.345517,0.500000,0.191033,0.070312,0.699219,256,W4A16,256.0,[]
2,awq_w4a16_wikitext_256,artifacts_compression_hw/awq_w4a16_wikitext_25...,awq,wikitext,0.266498,0.316406,0.216590,0.062500,0.468750,256,W4A16,256.0,[]


{'quality': 0.26649805587557607,
 'intent_accuracy': 0.31640625,
 'slot_micro_f1': 0.21658986175115208,
 'frame_exact_match': 0.0625,
 'valid_json_rate': 0.46875,
 'n': 256}

In [16]:
run_quantization_experiment('gptq_w4a16_massive_256')

Tokenizing: 100%|██████████| 256/256 [00:00<00:00, 761.19 examples/s]

2026-11-26T17:33:39.992062+0000 | reset | INFO - Compression lifecycle reset
2026-11-26T17:33:39.995314+0000 | from_modifiers | INFO - Creating recipe from modifiers
2026-11-26T17:33:40.046908+0000 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-11-26T17:33:40.047564+0000 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The

2026-11-26T17:34:07.401939+0000 | compress_modules | INFO - Quantizing model.layers.0.self_attn.q_proj using 256 samples


2026-11-26T17:34:09.316960+0000 | compress | METRIC - time 1.91s
2026-11-26T17:34:09.317873+0000 | compress | METRIC - error 0.47
2026-11-26T17:34:09.318729+0000 | compress | METRIC - GPU 0 | usage: 8.02% | total memory: 85 GB
2026-11-26T17:34:09.319232+0000 | compress | METRIC - Compressed module size: 33.947648 MB
2026-11-26T17:34:09.319795+0000 | compress_modules | INFO - Quantizing model.layers.0.self_attn.k_proj using 256 samples
2026-11-26T17:34:11.067142+0000 | compress | METRIC - time 1.75s
2026-11-26T17:34:11.068862+0000 | compress | METRIC - error 0.15
2026-11-26T17:34:11.069479+0000 | compress | METRIC - GPU 0 | usage: 8.02% | total memory: 85 GB
2026-11-26T17:34:11.069955+0000 | compress | METRIC - Compressed module size: 8.486912 MB
2026-11-26T17:34:11.070501+0000 | compress_modules | INFO - Quantizing model.layers.0.self_attn.v_proj using 256 samples
2026-11-26T17:34:12.805946+0000 | compress | METRIC - time 1.73s
2026-11-26T17:34:12.807864+0000 | compress | METRIC - erro

(2/37): Calibrating: 100%|██████████| 256/256 [00:09<00:00, 27.77it/s]

... (108 outputs left out)

2026-11-26T17:51:18.968582+0000 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2026-11-26T17:51:19.030580+0000 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`
2026-11-26T17:51:19.031432+0000 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 253it [00:17, 14.67it/s]
/opt/conda/lib/python3.11/site-packages/transformers/quantizers/auto.py:231: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.However, loading attributes (e.g. ['run_compressed']) will be overwritten with the one you passed to `from_pretrained`. The rest will be ignored.
  warnings.warn(warning_msg)
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  6.53it/s]
Decompressing model: 253it [00:23, 10.77it/s]
Оценка: 100%|██████████| 32/32 [06:29<00:00, 12.18s/it]


,run,path,method,calibration,quality,intent_accuracy,slot_micro_f1,frame_exact_match,valid_json_rate,n,scheme,num_samples,ignore
0,baseline,artifacts_compression_hw/baseline,dense,None,0.283519,0.375000,0.192037,0.105469,0.523438,256,NaN,NaN,NaN
1,awq_w4a16_massive_256,artifacts_compression_hw/awq_w4a16_massive_256...,awq,massive,0.345517,0.500000,0.191033,0.070312,0.699219,256,W4A16,256.0,[]
2,awq_w4a16_wikitext_256,artifacts_compression_hw/awq_w4a16_wikitext_25...,awq,wikitext,0.266498,0.316406,0.216590,0.062500,0.468750,256,W4A16,256.0,[]
3,gptq_w4a16_massive_256,artifacts_compression_hw/gptq_w4a16_massive_25...,gptq,massive,0.315687,0.394531,0.236842,0.109375,0.531250,256,W4A16,256.0,[]


{'quality': 0.3156866776315789,
 'intent_accuracy': 0.39453125,
 'slot_micro_f1': 0.23684210526315788,
 'frame_exact_match': 0.109375,
 'valid_json_rate': 0.53125,
 'n': 256}

In [17]:
run_quantization_experiment('smoothquant_w8a8_a05_massive_256')

Tokenizing: 100%|██████████| 256/256 [00:00<00:00, 735.55 examples/s]

2026-11-26T18:02:17.336311+0000 | reset | INFO - Compression lifecycle reset
2026-11-26T18:02:17.339497+0000 | from_modifiers | INFO - Creating recipe from modifiers
2026-11-26T18:02:17.340041+0000 | _infer_mappings_from_model | INFO - No SmoothQuantModifier.mappings provided, inferring from model...
2026-11-26T18:02:17.340489+0000 | get_layer_mappings_from_architecture | INFO - Architecture Qwen3ForCausalLM not found in mappings. Using default mappings: [LayerMap(balance_layers=['re:.*q_proj', 're:.*k_proj', 're:.*v_proj'], smooth_layers='re:.*input_layernorm'), LayerMap(balance_layers=['re:.*gate_proj', 're:.*up_proj'], smooth_layers='re:.*post_attention_layernorm')]


2026-11-26T18:02:18.437197+0000 | initialize | INFO - Compression lifecycle initialized for 2 modifiers
2026-11-26T18:02:18.438039+0000 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `SmoothQuantModifier`


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

2026-11-26T18:02:40.529521+0000 | _apply_smoothing | INFO - Smoothing with model.layers.0.input_layernorm
2026-11-26T18:02:40.532114+0000 | _apply_smoothing | INFO - Smoothing with model.layers.0.post_attention_layernorm



(2/37): Calibrating: 100%|██████████| 256/256 [00:01<00:00, 172.83it/s]

... (182 outputs left out)

(37/37): Propagating: 100%|██████████| 256/256 [00:01<00:00, 145.99it/s]


2026-11-26T18:21:11.864607+0000 | finalize | INFO - Compression lifecycle finalized for 2 modifiers
2026-11-26T18:21:11.918181+0000 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`
2026-11-26T18:21:11.919047+0000 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 253it [00:06, 36.38it/s]
/opt/conda/lib/python3.11/site-packages/transformers/quantizers/auto.py:231: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.However, loading attributes (e.g. ['run_compressed']) will be overwritten with the one you passed to `from_pretrained`. The rest will be ignored.
  warnings.warn(warning_msg)
Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


RuntimeError: Error(s) in loading state_dict for Linear:
	While copying the parameter named "weight", whose dimensions in the model are torch.Size([4096, 12288]) and whose dimensions in the checkpoint are torch.Size([4096, 12288]), an exception occurred : ('Only Tensors of floating point and complex dtype can require gradients',).

In [20]:
evaluate_saved_quantization_experiment(
    "smoothquant_w8a8_a05_massive_256"
)

Оценка: 100%|██████████| 32/32 [06:23<00:00, 11.99s/it]


,run,path,method,calibration,quality,intent_accuracy,slot_micro_f1,frame_exact_match,valid_json_rate,n,scheme,num_samples,ignore,smoothing_strength
0,baseline,artifacts_compression_hw/baseline,dense,None,0.283519,0.375000,0.192037,0.105469,0.523438,256,NaN,NaN,NaN,NaN
1,awq_w4a16_massive_256,artifacts_compression_hw/awq_w4a16_massive_256...,awq,massive,0.345517,0.500000,0.191033,0.070312,0.699219,256,W4A16,256.0,[],NaN
2,awq_w4a16_wikitext_256,artifacts_compression_hw/awq_w4a16_wikitext_25...,awq,wikitext,0.266498,0.316406,0.216590,0.062500,0.468750,256,W4A16,256.0,[],NaN
3,gptq_w4a16_massive_256,artifacts_compression_hw/gptq_w4a16_massive_25...,gptq,massive,0.315687,0.394531,0.236842,0.109375,0.531250,256,W4A16,256.0,[],NaN
4,smoothquant_w8a8_a05_massive_256,artifacts_compression_hw/smoothquant_w8a8_a05_...,smoothquant,massive,0.370537,0.507812,0.233261,0.167969,0.675781,256,W8A8,256.0,[],0.5


{'quality': 0.3705369195464363,
 'intent_accuracy': 0.5078125,
 'slot_micro_f1': 0.23326133909287258,
 'frame_exact_match': 0.16796875,
 'valid_json_rate': 0.67578125,
 'n': 256}

In [21]:
run_quantization_experiment('awq_w4a16_massive_256_lm_head_bf16')

Tokenizing: 100%|██████████| 256/256 [00:00<00:00, 727.61 examples/s]

2026-11-26T18:39:32.477979+0000 | reset | INFO - Compression lifecycle reset
2026-11-26T18:39:32.481105+0000 | from_modifiers | INFO - Creating recipe from modifiers


2026-11-26T18:39:32.532311+0000 | on_initialize | INFO - No AWQModifier.mappings provided, inferring from model...


Resolving mapping 1/4 (0 skipped): : 36it [00:00, 2016.33it/s]
Resolving mapping 2/4 (35 skipped): : 36it [00:00, 2927.22it/s]
Resolving mapping 3/4 (0 skipped): : 36it [00:00, 2380.61it/s]
Resolving mapping 4/4 (0 skipped): : 36it [00:00, 3011.65it/s]

2026-11-26T18:39:32.601233+0000 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-11-26T18:39:32.601755+0000 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `AWQModifier`



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The

2026-11-26T18:58:00.070467+0000 | finalize | INFO - Compression lifecycle finalized for 1 modifiers


2026-11-26T18:58:00.137535+0000 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`
2026-11-26T18:58:00.138343+0000 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 252it [00:07, 35.99it/s]
Оценка: 100%|██████████| 32/32 [04:36<00:00,  8.63s/it]


,run,path,method,calibration,quality,intent_accuracy,slot_micro_f1,frame_exact_match,valid_json_rate,n,scheme,num_samples,ignore,smoothing_strength
0,baseline,artifacts_compression_hw/baseline,dense,None,0.283519,0.375000,0.192037,0.105469,0.523438,256,NaN,NaN,NaN,NaN
1,awq_w4a16_massive_256,artifacts_compression_hw/awq_w4a16_massive_256...,awq,massive,0.345517,0.500000,0.191033,0.070312,0.699219,256,W4A16,256.0,[],NaN
2,awq_w4a16_wikitext_256,artifacts_compression_hw/awq_w4a16_wikitext_25...,awq,wikitext,0.266498,0.316406,0.216590,0.062500,0.468750,256,W4A16,256.0,[],NaN
3,gptq_w4a16_massive_256,artifacts_compression_hw/gptq_w4a16_massive_25...,gptq,massive,0.315687,0.394531,0.236842,0.109375,0.531250,256,W4A16,256.0,[],NaN
4,smoothquant_w8a8_a05_massive_256,artifacts_compression_hw/smoothquant_w8a8_a05_...,smoothquant,massive,0.370537,0.507812,0.233261,0.167969,0.675781,256,W8A8,256.0,[],0.5
5,awq_w4a16_massive_256_lm_head_bf16,artifacts_compression_hw/awq_w4a16_massive_256...,awq,massive,0.356131,0.468750,0.243513,0.093750,0.632812,256,W4A16,256.0,[lm_head],NaN


{'quality': 0.3561314870259481,
 'intent_accuracy': 0.46875,
 'slot_micro_f1': 0.2435129740518962,
 'frame_exact_match': 0.09375,
 'valid_json_rate': 0.6328125,
 'n': 256}

### Выводы

**Калибровочные данные.** Для одинакового AWQ W4A16 и 256 примеров калибровка на MASSIVE дала Quality `0.346`, а на WikiText — `0.266`. Для этой задачи важен домен калибровочных текстов: MASSIVE соответствует русским коротким запросам и требуемому JSON-формату.

**Конфигурации.** Проверены AWQ W4A16, GPTQ W4A16 и SmoothQuant W8A8 (`smoothing_strength=0.5`) на MASSIVE. Quality составило соответственно `0.346`, `0.316` и `0.371`; лучшей оказалась SmoothQuant W8A8. Для AWQ дополнительно `lm_head` был оставлен в BF16: Quality выросло с `0.346` до `0.356`, а Slot Micro-F1 — с `0.191` до `0.244`.

Наиболее заметным фактором оказался калибровочный датасет, затем — метод и точность активаций/весов. Квантизация не обязана монотонно ухудшать одну фиксированную метрику: при жадной генерации малое изменение весов может изменить дискретный JSON-ответ. Поэтому результаты интерпретируются только на этой фиксированной dev-выборке.

## Hard-label дистилляция (5 баллов)

Теперь реализуем hard-label дистилляцию знаний от учителя **Qwen3-32B** в бейзлайн.

### Задания
1) Измерьте качество учителя на dev и сравните с бейзлайном: насколько учитель сильнее? (1 балл)
2) Получите жадные ответы учителя на сэмпле из train. Используйте батч для ускорения при необходимости (1 балл)
3) Реализуйте код для обучения ученика на ответах учителя (1 балл)
4) Проведите эксперименты с обучением, получите модель ученика, отклоняющуюся на dev не более чем на 10% от метрики учителя (2 балла)


### Измерим качество учителя и сравним с бейзлайном (1 балл)

In [22]:
STUDENT_MODEL_ID = BASE_MODEL_ID
TEACHER_MODEL_ID = "Qwen/Qwen3-32B"

teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_ID)
if (
    tokenizer.get_vocab() != teacher_tokenizer.get_vocab()
    or tokenizer.chat_template != teacher_tokenizer.chat_template
    or tokenizer.eos_token_id != teacher_tokenizer.eos_token_id
    or tokenizer.pad_token_id != teacher_tokenizer.pad_token_id
):
    raise ValueError("Токенизаторы ученика и учителя различаются.")
del teacher_tokenizer

In [23]:
teacher = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL_ID, torch_dtype=torch.bfloat16,
    device_map=DEVICE,
).eval()
student_vocab_size = AutoConfig.from_pretrained(STUDENT_MODEL_ID).vocab_size
if teacher.config.vocab_size != student_vocab_size:
    raise ValueError("Размерности выходных логитов teacher и student различаются.")

Loading checkpoint shards: 100%|██████████| 17/17 [00:40<00:00,  2.35s/it]


In [26]:
### YOUR CODE HERE

TEACHER_GENERATE_BATCH_SIZE = 4
teacher_metrics, teacher_predictions = evaluate(
    teacher, tokenizer, dev_data, batch_size=TEACHER_GENERATE_BATCH_SIZE,
)
(ARTIFACTS / 'teacher_dev.json').write_text(
    json.dumps(teacher_metrics, ensure_ascii=False, indent=2), encoding='utf-8',
)
results = [old for old in results if old['run'] != 'teacher'] + [{
    'run': 'teacher', 'path': TEACHER_MODEL_ID, 'method': 'dense',
    'calibration': None, **teacher_metrics,
}]
display(pd.DataFrame(results))

Оценка: 100%|██████████| 64/64 [03:26<00:00,  3.23s/it]


,run,path,method,calibration,quality,intent_accuracy,slot_micro_f1,frame_exact_match,valid_json_rate,n,scheme,num_samples,ignore,smoothing_strength
0,baseline,artifacts_compression_hw/baseline,dense,None,0.283519,0.375000,0.192037,0.105469,0.523438,256,NaN,NaN,NaN,NaN
1,awq_w4a16_massive_256,artifacts_compression_hw/awq_w4a16_massive_256...,awq,massive,0.345517,0.500000,0.191033,0.070312,0.699219,256,W4A16,256.0,[],NaN
2,awq_w4a16_wikitext_256,artifacts_compression_hw/awq_w4a16_wikitext_25...,awq,wikitext,0.266498,0.316406,0.216590,0.062500,0.468750,256,W4A16,256.0,[],NaN
3,gptq_w4a16_massive_256,artifacts_compression_hw/gptq_w4a16_massive_25...,gptq,massive,0.315687,0.394531,0.236842,0.109375,0.531250,256,W4A16,256.0,[],NaN
4,smoothquant_w8a8_a05_massive_256,artifacts_compression_hw/smoothquant_w8a8_a05_...,smoothquant,massive,0.370537,0.507812,0.233261,0.167969,0.675781,256,W8A8,256.0,[],0.5
5,awq_w4a16_massive_256_lm_head_bf16,artifacts_compression_hw/awq_w4a16_massive_256...,awq,massive,0.356131,0.468750,0.243513,0.093750,0.632812,256,W4A16,256.0,[lm_head],NaN
6,teacher,Qwen/Qwen3-32B,dense,None,0.584408,0.785156,0.383659,0.316406,0.976562,256,NaN,NaN,NaN,NaN


### Получим ответы учителя на трейне (1 балл)

Не забывайте выгружать модели и очищать память.

In [27]:
DISTILL_SAMPLES = 128

def collect_teacher_answers(
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizerBase,
    rows: Dataset,
    output_path: str | Path,
) -> Path:
    """Генерируем ответы учителя батчами и сохраняем их в answers.jsonl.

    Args:
        model: модель-учитель.
        tokenizer: общий токенизатор учителя и ученика.
        rows: примеры train с полями `id`, `text`, `gold`.
        output_path: пустой каталог для ответов.

    Returns:
        Каталог с answers.jsonl; в каждой строке `id`, `text`, `gold`,
        `teacher_answer`, `prompt_ids`, `answer_ids`.
    """
    output_path = ensure_empty_dir(output_path)
    records = []
    row_list = list(rows)

    ### YOUR CODE HERE
    for start in tqdm(
        range(0, len(row_list), TEACHER_GENERATE_BATCH_SIZE),
        desc='Ответы учителя',
    ):
        batch_rows = row_list[start:start + TEACHER_GENERATE_BATCH_SIZE]
        prompt_ids = [encode_prompt(tokenizer, row['text']) for row in batch_rows]
        answer_ids = generate_answer_ids(model, tokenizer, prompt_ids)
        answers = tokenizer.batch_decode(answer_ids, skip_special_tokens=True)
        for row, prompt, answer, answer_token_ids in zip(
            batch_rows, prompt_ids, answers, answer_ids,
        ):
            records.append({
                'id': row['id'],
                'text': row['text'],
                'gold': row['gold'],
                'teacher_answer': answer,
                'prompt_ids': prompt,
                'answer_ids': answer_token_ids,
            })


    if not records:
        raise ValueError("Учитель не вернул ни одного ответа.")
    (output_path / "answers.jsonl").write_text(
        "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in records),
        encoding="utf-8",
    )
    print(f"Сохранено ответов: {len(records)} / {len(row_list)}")
    return output_path

In [28]:
distill_rows = train_data.shuffle(seed=SEED).select(
    range(min(DISTILL_SAMPLES, len(train_data)))
)
TEACHER_ANSWERS = collect_teacher_answers(
    teacher, tokenizer, distill_rows, ARTIFACTS / "teacher_answers"
)
del teacher
clean_memory()

Ответы учителя: 100%|██████████| 32/32 [01:31<00:00,  2.87s/it]


Сохранено ответов: 128 / 128


### Реализация пайплайна обучения ученика (1 балл)

Полное дообучение Qwen3-8B на A100 80GB проходит на грани по памяти. Для решения можно включать gradient checkpointing, уменьшать batch или брать меньшую модель того же семейства.

In [14]:
class TeacherAnswerDataset(torch.utils.data.Dataset):
    """Примеры для обучения ученика на ответах учителя."""

    def __init__(self, path: str | Path) -> None:
        """

        Args:
            path: каталог с answers.jsonl из collect_teacher_answers.
        """
        self.rows = [] # Список примеров из файла

        ### YOUR CODE HERE
        ### Fill rows
        answers_path = Path(path) / 'answers.jsonl'
        for line in answers_path.read_text(encoding='utf-8').splitlines():
            row = json.loads(line)
            if not row['prompt_ids'] or not row['answer_ids']:
                continue
            self.rows.append(row)
        if not self.rows:
            raise ValueError('В answers.jsonl нет пригодных ответов учителя.')


    def __len__(self) -> int:
        """
            Число строк в answers.jsonl.
        """
        ### YOUR CODE HERE
        return len(self.rows)

    def __getitem__(self, index: int) -> dict[str, list[int]]:
        """
        Args:
            index: номер примера.

        Returns:
            Словарь с `input_ids` (промпт + ответ), `attention_mask` и `labels`,
            токены промпта маскируем значением -100.
        """
        row = self.rows[index]
        input_ids = row['prompt_ids'] + row['answer_ids']
        labels = [-100] * len(row['prompt_ids']) + row['answer_ids']
        return {
            "input_ids": input_ids,
            "attention_mask": [1] * len(input_ids),
            "labels": labels,
        }

def collate_teacher_answers(features: list[dict[str, list[int]]]) -> dict[str, torch.Tensor]:
    """Паддим примеры справа до общей длины батча.

    Args:
        features: примеры из TeacherAnswerDataset.

    Returns:
        Тензоры `input_ids`, `attention_mask` и `labels` формы [batch, length];
        паддинг в `labels` равен -100.
    """
    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        pad_id = tokenizer.eos_token_id
    width = max(len(feature["input_ids"]) for feature in features)
    batch = {"input_ids": [], "attention_mask": [], "labels": []}
    for feature in features:
        pad = width - len(feature["input_ids"])
        batch["input_ids"].append(feature["input_ids"] + [pad_id] * pad)
        batch["attention_mask"].append(feature["attention_mask"] + [0] * pad)
        batch["labels"].append(feature["labels"] + [-100] * pad)
    return {key: torch.tensor(value) for key, value in batch.items()}

In [15]:
STUDENT_EXPERIMENTS = {
    'student_sft_128_adafactor_lr1e5_e3': {
        'learning_rate': 1e-5,
        'num_train_epochs': 3,
        'per_device_train_batch_size': 1,
        'gradient_accumulation_steps': 8,
        'optim': 'adafactor',
    },
    'student_sft_128_adafactor_lr2e5_e3': {
        'learning_rate': 2e-5,
        'num_train_epochs': 3,
        'per_device_train_batch_size': 1,
        'gradient_accumulation_steps': 8,
        'optim': 'adafactor',
    },
}

def run_student_experiment(name: str) -> dict[str, float]:
    if name not in STUDENT_EXPERIMENTS:
        raise KeyError(f'Нет конфигурации {name}.')
    config = STUDENT_EXPERIMENTS[name]
    run_dir = ensure_empty_dir(ARTIFACTS / name)
    train_dataset = TeacherAnswerDataset(TEACHER_ANSWERS)
    (run_dir / 'config.json').write_text(
        json.dumps({
            'student_model': STUDENT_MODEL_ID, 'teacher_model': TEACHER_MODEL_ID,
            'teacher_answers': str(TEACHER_ANSWERS),
            'train_examples': len(train_dataset), **config,
        }, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )

    set_seed(SEED)
    student = AutoModelForCausalLM.from_pretrained(
        STUDENT_MODEL_ID, torch_dtype=torch.bfloat16,
    )
    student.config.use_cache = False
    student.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={'use_reentrant': False},
    )
    args = TrainingArguments(
        output_dir=str(run_dir / 'checkpoints'),
        per_device_train_batch_size=config['per_device_train_batch_size'],
        gradient_accumulation_steps=config['gradient_accumulation_steps'],
        learning_rate=config['learning_rate'],
        num_train_epochs=config['num_train_epochs'],
        optim=config['optim'],
        bf16=True,
        gradient_checkpointing=True,
        logging_strategy='steps', logging_steps=1,
        save_strategy='epoch', save_total_limit=1,
        report_to=[], remove_unused_columns=False,
        dataloader_num_workers=0, seed=SEED,
    )
    trainer = Trainer(
        model=student, args=args, train_dataset=train_dataset,
        data_collator=collate_teacher_answers,
    )
    try:
        train_result = trainer.train()
        trainer.save_model(run_dir / 'model')
        tokenizer.save_pretrained(run_dir / 'model')
        (run_dir / 'train_metrics.json').write_text(
            json.dumps(train_result.metrics, ensure_ascii=False, indent=2),
            encoding='utf-8',
        )
        trainer.state.save_to_json(run_dir / 'trainer_state.json')
    finally:
        del trainer, student
        clean_memory()

    metrics, predictions = evaluate_checkpoint(run_dir / 'model', dev_data)
    (run_dir / 'metrics.json').write_text(
        json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8',
    )
    save_predictions(run_dir / 'predictions.jsonl', dev_data, predictions)
    row = {'run': name, 'path': str(run_dir / 'model'), 'method': 'hard_label_sft',
           'calibration': None, **config, **metrics}
    global results
    results = [old for old in results if old['run'] != name] + [row]
    pd.DataFrame(results).to_json(
        ARTIFACTS / 'distillation_results.json',
        orient='records', force_ascii=False, indent=2,
    )
    display(pd.DataFrame(results))
    return metrics

In [16]:
BASELINE_PATH = ARTIFACTS / "baseline"
tokenizer = AutoTokenizer.from_pretrained(BASELINE_PATH)

STUDENT_MODEL_ID = BASE_MODEL_ID
TEACHER_MODEL_ID = "Qwen/Qwen3-32B"
TEACHER_ANSWERS = ARTIFACTS / "teacher_answers"

baseline_metrics = json.loads(
    (ARTIFACTS / "baseline_dev.json").read_text(encoding="utf-8")
)
results = [{
    "run": "baseline",
    "path": str(BASELINE_PATH),
    "method": "dense",
    "calibration": None,
    **baseline_metrics,
}]

In [19]:
sft_metrics = run_student_experiment(
    "student_sft_128_adafactor_lr1e5_e3"
)

Loading checkpoint shards: 100%|██████████| 5/5 [00:00<00:00, 117.87it/s]


Step,Training Loss
1,0.478200
2,0.644800
3,0.793700
4,0.191400
5,0.236000
6,0.354800
7,0.213600
8,0.197500
9,0.276300
10,0.299700


Оценка:   0%|          | 0/32 [00:00<?, ?it/s]`generation_config` default values have been modified to match model-specific defaults: {'do_sample': True, 'temperature': 0.6, 'top_k': 20, 'top_p': 0.95, 'bos_token_id': 151643}. If this is not desired, please set these values explicitly.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Оценка: 100%|██████████| 32/32 [00:53<00:00,  1.67s/it]


,run,path,method,calibration,quality,intent_accuracy,slot_micro_f1,frame_exact_match,valid_json_rate,n,learning_rate,num_train_epochs,per_device_train_batch_size,gradient_accumulation_steps,optim
0,baseline,artifacts_compression_hw/baseline,dense,None,0.283519,0.375,0.192037,0.105469,0.523438,256,NaN,NaN,NaN,NaN,NaN
1,student_sft_128_adafactor_lr1e5_e3,artifacts_compression_hw/student_sft_128_adafa...,hard_label_sft,None,0.574640,0.750,0.399281,0.277344,0.988281,256,0.00001,3.0,1.0,8.0,adafactor


### Добейтесь максимального качества ученика. Цель — не более 10% отклонения от учителя (2 балла)
Пробуйте менять гиперпараметры, число сэмплов, модели ученика (при условии размера модели ученика не более 8B).

In [20]:
sft_metrics_lr2e5 = run_student_experiment(
    "student_sft_128_adafactor_lr2e5_e3"
)

Loading checkpoint shards: 100%|██████████| 5/5 [00:00<00:00, 115.12it/s]


Step,Training Loss
1,0.478200
2,1.640900
3,0.794600
4,0.532700
5,0.367500
6,0.433900
7,0.369700
8,0.228600
9,0.343600
10,0.355200


Оценка: 100%|██████████| 32/32 [00:53<00:00,  1.66s/it]


,run,path,method,calibration,quality,intent_accuracy,slot_micro_f1,frame_exact_match,valid_json_rate,n,learning_rate,num_train_epochs,per_device_train_batch_size,gradient_accumulation_steps,optim
0,baseline,artifacts_compression_hw/baseline,dense,None,0.283519,0.375000,0.192037,0.105469,0.523438,256,NaN,NaN,NaN,NaN,NaN
1,student_sft_128_adafactor_lr1e5_e3,artifacts_compression_hw/student_sft_128_adafa...,hard_label_sft,None,0.574640,0.750000,0.399281,0.277344,0.988281,256,0.00001,3.0,1.0,8.0,adafactor
2,student_sft_128_adafactor_lr2e5_e3,artifacts_compression_hw/student_sft_128_adafa...,hard_label_sft,None,0.601843,0.765625,0.438061,0.304688,0.976562,256,0.00002,3.0,1.0,8.0,adafactor


### Выводы

Teacher Qwen3-32B получил Quality `0.584`, что в `2.06x` выше baseline Qwen3-8B (`0.284`). Для обучения были сгенерированы жадные ответы teacher на 128 примерах MASSIVE train батчем 4; проверка проводилась на отдельной выборке MASSIVE validation из 256 примеров.

Ученика Qwen3-8B обучали три эпохи с batch size 1, gradient accumulation 8, gradient checkpointing и Adafactor. При `lr=1e-5` Quality составило `0.575` — `98.3%` качества teacher. При изменении только learning rate на `2e-5` Quality выросло до `0.602`: Intent Accuracy `0.766`, Slot Micro-F1 `0.438`, Frame Exact Match `0.305`, valid JSON rate `0.977`.

Лучшая конфигурация — `student_sft_128_adafactor_lr2e5_e3`. Она превышает Quality teacher на данной dev-выборке (`103.0%`), поэтому требование отклонения не более 10% выполнено. Рост относительно baseline связан прежде всего с освоением строгого формата: valid JSON rate вырос с `0.523` до `0.977`.